# Pipeline de Poda Estruturada — MLP no CIFAR-10

**Objetivo:** Comparar **Poda Estruturada Local** vs **Poda Estruturada Global** utilizando saliência **L1** e **l2** numa rede MLP (*Multi-Layer Perceptron*) totalmente ligada, com avaliação automática para múltiplos níveis de esparsidade.

Este notebook implementa:

1. **Configuração e Importações** — Configuração do dispositivo e monitorização da VRAM.
2. **Poda Estruturada Global** — Classificação e remoção de neurónios em toda a rede com base na sua importância.
3. **Ajuste Fino (*Fine-Tuning*)** — Recuperação iterativa do desempenho após a poda (SGD com Momentum).
4. **Pipeline Automatizado** — Comparação sistemática entre diferentes níveis de esparsidade.
5. **Visualização** — Gráficos comparativos (Precisão, Latência e Taxa de Compressão).


---
## 1. Setup e Importações

Carregamento de todas as dependências, importação das arquiteturas e métricas de `utils.py`, e configuração do dispositivo de computação com tracking inicial de memória VRAM.

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import numpy as np
import random
import copy
import time
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset

# ── Importações do utils.py ──
from utils import (
    CIFAR10MLP,
    evaluate_model,
    compute_metrics,
    train_model,
    classes
)

# ── Reprodutibilidade ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dispositivo (foco em CUDA) ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    vram_initial = torch.cuda.memory_allocated(device) / (1024**2)
    print(f"GPU: {torch.cuda.get_device_name(device)}")
    print(f"VRAM alocada inicialmente: {vram_initial:.2f} MB")
else:
    print("AVISO: CUDA não disponível. Os resultados de VRAM serão 0.")

Device: cpu
AVISO: CUDA não disponível. Os resultados de VRAM serão 0.


### Carregamento de Dados — Split Determinístico

Para garantir uma comparação justa entre as experiências de poda, reutilizamos o split
determinístico treino/validação gerado durante o treino do baseline.

In [13]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

full_trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

# Reutilizar o split determinístico guardado
split = torch.load("data/cifar10_split.pt", weights_only=False)
train_indices = split["train_indices"]
val_indices = split["val_indices"]

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

batch_size = 64
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Train: {len(trainset):,} | Val: {len(valset):,} | Test: {len(testset):,}")

Train: 42,500 | Val: 7,500 | Test: 10,000


### Carregamento do Modelo Baseline

Carregamento do MLP pré-treinado no CIFAR-10. Este modelo serve como ponto de referência
para todas as experiências de poda subsequentes.

In [14]:
models_dir = Path('model_dir')
PATH_MLP = models_dir / 'cifar_mlp.pt'

baseline_model = torch.load(PATH_MLP, map_location=torch.device('cpu'), weights_only=False) 
baseline_model = baseline_model.to(device)
baseline_model.eval()

print(baseline_model)
print(f"\nTotal de parâmetros: {sum(p.numel() for p in baseline_model.parameters()):,}")

if device.type == "cuda":
    vram_model = torch.cuda.memory_allocated(device) / (1024**2)
    print(f"VRAM após carregar modelo: {vram_model:.2f} MB")

CIFAR10MLP(
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=2048, bias=True)
    (2): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.3, inplace=False)
    (5): Linear(in_features=2048, out_features=1024, bias=True)
    (6): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=1024, out_features=512, bias=True)
    (10): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): ReLU()
    (12): Dropout(p=0.2, inplace=False)
    (13): Linear(in_features=512, out_features=10, bias=True)
  )
)

Total de parâmetros: 8,928,778


---
## 2. Funções de Saliência para Estimativa de Importância dos Neurónios

A saliência de cada neurónio é calculada com base na norma dos seus pesos.
Neurónios com menor saliência são considerados menos importantes e são candidatos à remoção.

Para o neurónio $j$ na camada com pesos $W$:

**Norma L1:**  $S_j = \sum_i |w_{ij}|$

**Norma L2:**  $S_j = \sqrt{\sum_i w_{ij}^2}$

In [15]:
# Saliency Functions — Neuron Importance Estimation

def compute_l1_saliency(layer: nn.Linear) -> torch.Tensor:
    """
    Saliência baseada na Norma L1 dos pesos de saída.
    
    Para o neurónio j:  S_j = Σ_i |w_{ij}|
    
    Neurónios com menor saliência são considerados menos importantes.
    """
    W = layer.weight.data  # Shape: [out_features, in_features]
    saliency = torch.sum(torch.abs(W), dim=1)  # L1 por neurónio
    return saliency


def compute_l2_saliency(layer: nn.Linear) -> torch.Tensor:
    """
    Saliência baseada na Norma L2 dos pesos de saída.
    
    Para o neurónio j:  S_j = sqrt(Σ_i w_{ij}²)
    """
    W = layer.weight.data
    saliency = torch.norm(W, p=2, dim=1)
    return saliency

def compute_l1_saliency(layer: nn.Linear) -> torch.Tensor:
    return torch.sum(torch.abs(layer.weight.data), dim=1)

def compute_l2_saliency(layer: nn.Linear) -> torch.Tensor:
    return torch.norm(layer.weight.data, p=2, dim=1)

def compute_taylor_saliency(model: nn.Module, layer: nn.Module, calibration_loader: DataLoader, device: torch.device) -> torch.Tensor:
    """Implementação da pág. 6 do relatório: S_j = |(dL/da_j) * a_j|"""
    activations = []
    gradients = []
    
    # Hooks para extrair ativações e gradientes internamente
    def f_hook(module, input, output): activations.append(output.detach())
    def b_hook(module, grad_in, grad_out): gradients.append(grad_out[0].detach())
    
    h_f = layer.register_forward_hook(f_hook)
    h_b = layer.register_full_backward_hook(b_hook)
    
    # Forward e Backward com apenas 1 batch de treino (Calibração)
    model.eval()
    images, labels = next(iter(calibration_loader))
    outputs = model(images.to(device))
    loss = nn.CrossEntropyLoss()(outputs, labels.to(device))
    model.zero_grad()
    loss.backward()
    
    h_f.remove()
    h_b.remove()
    
    # Saliência baseada em Taylor (Média ao longo do lote)
    act = activations[0]
    grad = gradients[0]
    return (act * grad).abs().mean(dim=0)

print("Funções de saliência definidas: compute_l1_saliency, compute_l2_saliency")

Funções de saliência definidas: compute_l1_saliency, compute_l2_saliency


### Reconstrução Física das Camadas (In-Place Tensor Manipulation)

Em vez de mascarar pesos a zero, a arquitetura da rede é **fisicamente reconstruída**.
Isto garante que as matrizes de pesos continuam a multiplicar corretamente após a poda,
ajustando dinamicamente `in_features` e `out_features` das camadas subsequentes.

Isto reduz:
- Contagem de parâmetros
- Consumo de memória
- Latência de inferência

In [16]:
def apply_layer_pruning_physical(
    classifier: nn.Sequential,
    layer_idx: int,
    indices_to_keep: torch.Tensor
):
    """
    Remove fisicamente neurónios de uma camada Linear alvo e propaga as
    alterações dimensionais para as camadas adjacentes (BatchNorm + próxima Linear).
    
    Esta é PODA ESTRUTURADA REAL — a arquitetura é fisicamente reconstruída,
    reduzindo contagem de parâmetros, memória e latência de inferência.
    
    Garante o ajuste dinâmico de in_features e out_features para que as
    matrizes continuem a multiplicar corretamente.
    
    Args:
        classifier: nn.Sequential contendo todas as camadas
        layer_idx:  Índice da camada Linear a podar
        indices_to_keep: Tensor 1D com índices dos neurónios a reter
    """
    layer = classifier[layer_idx]
    indices_to_keep = torch.sort(indices_to_keep).values
    num_to_keep = len(indices_to_keep)
    
    # ── 1. Podar a dimensão de saída da camada alvo ──
    # Selecionar apenas as linhas (neurónios) a manter
    layer.weight = nn.Parameter(layer.weight.data[indices_to_keep, :])
    if layer.bias is not None:
        layer.bias = nn.Parameter(layer.bias.data[indices_to_keep])
    layer.out_features = num_to_keep
    
    # ── 2. Propagar para camadas downstream (BatchNorm) ──
    next_linear_idx = None
    for i in range(layer_idx + 1, len(classifier)):
        if isinstance(classifier[i], nn.BatchNorm1d):
            bn = classifier[i]
            bn.weight = nn.Parameter(bn.weight.data[indices_to_keep])
            bn.bias = nn.Parameter(bn.bias.data[indices_to_keep])
            bn.running_mean = bn.running_mean[indices_to_keep]
            bn.running_var = bn.running_var[indices_to_keep]
            bn.num_features = num_to_keep
        elif isinstance(classifier[i], nn.Linear):
            next_linear_idx = i
            break
    
    # ── 3. Ajustar a dimensão de entrada da próxima camada Linear ──
    # Isto é CRÍTICO para a validade arquitetural: garante que a
    # multiplicação de matrizes continua a funcionar corretamente
    if next_linear_idx is not None:
        next_layer = classifier[next_linear_idx]
        next_layer.weight = nn.Parameter(
            next_layer.weight.data[:, indices_to_keep]
        )
        next_layer.in_features = num_to_keep

print("Função apply_layer_pruning_physical definida.")

Função apply_layer_pruning_physical definida.


### Poda Estruturada Local (Local Structured Pruning)

Na poda **local**, cada camada oculta perde exatamente a mesma fração de neurónios
(`prune_ratio`). O ranking é feito **independentemente** por camada.

In [17]:
def prune_mlp_local(
    model: nn.Module,
    prune_ratio: float,
    saliency_fn=compute_l1_saliency
) -> nn.Module:
    """
    PODA ESTRUTURADA LOCAL: Aplica o mesmo rácio de poda independentemente
    a cada camada Linear oculta. Cada camada perde exatamente `prune_ratio`
    fração dos seus neurónios.
    
    Args:
        model: O modelo MLP original
        prune_ratio: Fração de neurónios a remover por camada (0.0 a 1.0)
        saliency_fn: Função que calcula scores de saliência por neurónio
        
    Returns:
        Novo modelo podado (deep copy — o original mantém-se intacto)
    """
    if prune_ratio >= 1.0:
        prune_ratio = prune_ratio / 100.0
    
    pruned_model = copy.deepcopy(model)
    classifier = pruned_model.classifier
    
    # Identificar todas as camadas Linear (excluindo a de saída)
    linear_indices = [
        i for i, layer in enumerate(classifier)
        if isinstance(layer, nn.Linear)
    ]
    hidden_linear_indices = linear_indices[:-1]
    
    print(f"  Poda Local (ratio={prune_ratio:.2f}) nas camadas: "
          f"{hidden_linear_indices}")
    
    for idx in hidden_linear_indices:
        layer = classifier[idx]
        saliency = saliency_fn(layer)
        n_neurons = saliency.shape[0]
        n_keep = max(1, int(n_neurons * (1 - prune_ratio)))
        
        # Ranking e seleção dos melhores neurónios
        indices_to_keep = torch.argsort(
            saliency, descending=True
        )[:n_keep]
        
        print(f"    Camada[{idx}]: {n_neurons} → {n_keep} neurónios "
              f"(removidos {n_neurons - n_keep})")
        
        apply_layer_pruning_physical(classifier, idx, indices_to_keep)
    
    return pruned_model.to(device)

print("Função prune_mlp_local definida.")

Função prune_mlp_local definida.


---
## 2. Lógica de Global Structured Pruning

Na poda **global**, a saliência é calculada para **TODOS** os neurónios de **TODAS**
as camadas ocultas, criando um **ranking unificado**. O bottom `prune_ratio` é removido
globalmente — independentemente da camada a que pertençam.

**Vantagem sobre a poda local:**
Preserva mais capacidade nas camadas que são realmente importantes, enquanto
comprime agressivamente camadas com neurónios redundantes.

$$\text{Threshold} = \text{kth\_smallest}\left(\bigcup_{l \in \text{hidden}} \{S_j^{(l)}\}_{j=1}^{n_l},\; k = \lfloor N_{\text{total}} \times r \rfloor\right)$$

In [18]:
def prune_mlp_global(
    model: nn.Module,
    prune_ratio: float,
    saliency_fn=compute_l1_saliency
) -> nn.Module:
    """
    PODA ESTRUTURADA GLOBAL: Calcula a saliência de TODOS os neurónios de TODAS
    as camadas ocultas do MLP, reúne-os num ranking global e remove a percentagem
    global estipulada de parâmetros menos importantes, independentemente da camada
    a que pertençam (mantendo a validade arquitetural da rede).
    
    Ao contrário da poda local, isto significa que algumas camadas podem ser podadas
    mais agressivamente que outras, com base na distribuição real de importância.
    
    Args:
        model: O modelo MLP original
        prune_ratio: Fração do total de neurónios ocultos a remover (0.0 a 1.0)
        saliency_fn: Função que calcula scores de saliência por neurónio.
                     ──────────────────────────────────────────────────
                     HOOK DE INTEGRAÇÃO: Esta função pode ser substituída
                     por métodos baseados em gradientes (Taylor 1ª ordem,
                     Optimal Brain Damage) fornecendo uma função com
                     assinatura compatível: fn(layer) -> Tensor[n_neurons]
                     ──────────────────────────────────────────────────
        
    Returns:
        Novo modelo podado (deep copy — o original mantém-se intacto)
    """
    if prune_ratio >= 1.0:
        prune_ratio = prune_ratio / 100.0
    
    pruned_model = copy.deepcopy(model)
    classifier = pruned_model.classifier
    
    # Identificar camadas Linear ocultas
    linear_indices = [
        i for i, layer in enumerate(classifier)
        if isinstance(layer, nn.Linear)
    ]
    hidden_linear_indices = linear_indices[:-1]
    
    # ── Passo 1: Recolher saliências num pool global ──
    all_saliencies = []
    layer_boundaries = []  # Rastrear onde começam/acabam os scores de cada camada
    offset = 0
    
    for idx in hidden_linear_indices:
        scores = saliency_fn(classifier[idx])
        all_saliencies.append(scores)
        n = scores.numel()
        layer_boundaries.append((offset, offset + n, idx))
        offset += n
    
    global_pool = torch.cat(all_saliencies)
    total_neurons = global_pool.numel()
    num_to_prune = int(total_neurons * prune_ratio)
    
    # ── Passo 2: Calcular o limiar global ──
    # ──────────────────────────────────────────────────────────────
    # HOOK PARA SALIÊNCIA BASEADA EM GRADIENTES:
    # Para métodos como Taylor de 1ª ordem ou OBD, substituir
    # `saliency_fn` por uma função que incorpore info de gradientes.
    # O mecanismo de ranking global permanece idêntico — apenas a
    # computação do score por neurónio muda.
    # ──────────────────────────────────────────────────────────────
    if num_to_prune > 0 and num_to_prune < total_neurons:
        global_threshold = torch.kthvalue(
            global_pool, num_to_prune
        ).values.item()
    else:
        global_threshold = -float('inf')
    
    print(f"  Poda Global → Total neurónios: {total_neurons} | "
          f"Alvo remoção: {num_to_prune} | "
          f"Limiar: {global_threshold:.6f}")
    
    # ── Passo 3: Podar cada camada contra o limiar global ──
    for start, end, idx in layer_boundaries:
        layer = classifier[idx]
        saliency = saliency_fn(layer)
        
        # Manter neurónios estritamente acima do limiar global
        indices_to_keep = torch.where(saliency > global_threshold)[0]
        
        # Guardrail de segurança: manter sempre pelo menos 1 neurónio
        # por camada para evitar colapso dimensional
        if len(indices_to_keep) == 0:
            indices_to_keep = torch.argmax(saliency).unsqueeze(0)
        
        original_size = layer.out_features
        print(f"    Camada[{idx}]: {original_size} → "
              f"{len(indices_to_keep)} neurónios "
              f"(removidos {original_size - len(indices_to_keep)}, "
              f"{(1 - len(indices_to_keep)/original_size)*100:.1f}%)")
        
        apply_layer_pruning_physical(classifier, idx, indices_to_keep)
    
    return pruned_model.to(device)

print("Função prune_mlp_global definida.")

Função prune_mlp_global definida.


---
## 3. Ciclo de Fine-Tuning (Iterative Post-Training)

Após o corte in-place dos tensores, os pesos sobreviventes precisam de ser
recalibrados. Este fine-tuning rápido utiliza **SGD com Momentum** (estável
para recalibração pós-poda) durante poucas epochs para recuperar a exatidão.

In [19]:
def fine_tune(
    model: nn.Module,
    trainloader: DataLoader,
    valloader: DataLoader,
    epochs: int = 5,
    lr: float = 1e-4,  # Revertido para 1e-4 (mais estável para pós-poda)
    optimizer_type: str = "sgd",
    device: torch.device = device
) -> nn.Module:
    """
    Fine-tuning rápido pós-poda para recuperar a estabilidade e exatidão
    dos pesos sobreviventes, agora com barra de progresso tqdm integrada por lote.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    if optimizer_type.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(
            model.parameters(), lr=lr, momentum=0.9
        )
    
    print(f"  Fine-tuning: {epochs} epochs | LR={lr} | "
          f"Optimizer={optimizer_type.upper()}\")")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        # Barra de progresso iterativa por lote, como no guião antigo
        progress_bar = tqdm(
            trainloader, 
            desc=f"    Epoch [{epoch+1}/{epochs}]", 
            leave=False
        )
        
        for images, labels in progress_bar:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())
        
        train_loss = running_loss / len(trainloader)
        
        # ── Validação ──
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in valloader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_loss /= len(valloader)
        val_acc = 100.0 * correct / total
        
        print(f"    Epoch [{epoch+1}/{epochs}] Terminado -> "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}%")
    
    return model

---
## 4. Pipeline de Avaliação Automatizada

### Função de Avaliação Abrangente

Para cada modelo (baseline ou podado), regista:
- **a)** Exatidão (Accuracy) de teste
- **b)** Contagem de parâmetros ativos
- **c)** Tempo real de inferência (Latência em segundos)
- **d)** Uso de VRAM no dispositivo (se aplicável)

In [20]:
def evaluate_comprehensive(
    model: nn.Module,
    testloader: DataLoader,
    device: torch.device,
    num_inference_runs: int = 3
) -> dict:
    """
    Avaliação abrangente com todas as métricas requeridas:
      a) Exatidão de teste (accuracy)
      b) Contagem de parâmetros ativos
      c) Latência total de inferência (segundos)
      d) Uso de VRAM (MB, se CUDA)
    
    Args:
        model: Modelo a avaliar
        testloader: DataLoader de teste
        device: Dispositivo de computação
        num_inference_runs: Nº de runs para média da latência
        
    Returns:
        Dicionário com accuracy, param_count, latency_seconds, vram_mb
    """
    model = model.to(device)
    model.eval()
    
    # ── Contagem de Parâmetros ──
    total_params = sum(p.numel() for p in model.parameters())
    
    # ── Reset VRAM tracking ──
    vram_mb = 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()
    
    # ── Exatidão (usando evaluate_model do utils.py) ──
    y_true, y_pred, _ = evaluate_model(model, testloader, device)
    metrics = compute_metrics(y_true, y_pred)
    accuracy = metrics["accuracy"]
    
    # ── Latência de Inferência ──
    latencies = []
    for _ in range(num_inference_runs):
        if device.type == "cuda":
            torch.cuda.synchronize()
        
        start_time = time.perf_counter()
        
        with torch.no_grad():
            for images, labels in testloader:
                images = images.to(device)
                _ = model(images)
        
        if device.type == "cuda":
            torch.cuda.synchronize()
        
        end_time = time.perf_counter()
        latencies.append(end_time - start_time)
    
    avg_latency = np.mean(latencies)
    
    # ── Pico VRAM ──
    if device.type == "cuda":
        vram_mb = torch.cuda.max_memory_allocated(device) / (1024**2)
    
    return {
        "accuracy": accuracy,
        "param_count": total_params,
        "latency_seconds": avg_latency,
        "vram_mb": vram_mb
    }

print("Função evaluate_comprehensive definida.")

Função evaluate_comprehensive definida.


In [ ]:
def run_pruning_pipeline(
    model: nn.Module,
    prune_fn,           # prune_mlp_local ou prune_mlp_global
    saliency_fn,        # compute_l1_saliency, compute_l2_saliency, etc.
    target_ratio: float, # O rácio final pretendido (ex: 0.30 para 30%)
    strategy: str = "one-shot", 
    epochs_per_step: int = 5,
    lr: float = 1e-4,
    trainloader = trainloader,
    valloader = valloader,
    device = device
) -> nn.Module:
    """
    Controlador do Passo 3: Executa tanto a Poda One-Shot como a Poda Iterativa,
    corrigido para recalcular os rácios relativos em cada sub-passo físico.
    """
    current_model = copy.deepcopy(model)
    
    if strategy.lower() == "one-shot":
        print(f"\n🚀 A iniciar Poda ONE-SHOT (Alvo: {target_ratio*100:.0f}%)")
        current_model = prune_fn(current_model, target_ratio, saliency_fn)
        current_model = fine_tune(
            current_model, trainloader, valloader, 
            epochs=epochs_per_step, lr=lr, device=device
        )
        
    elif strategy.lower() == "iterative":
        print(f"\n🔄 A iniciar Poda ITERATIVA (Alvo Final: {target_ratio*100:.0f}%)")
        num_steps = 3
        
        # Quantidade absoluta a remover do total original em cada passo (ex: 10% se o alvo for 30%)
        absolute_step_chunk = target_ratio / num_steps
        
        pre_ft = None
        
        for step in range(num_steps):
            # Calcular quanto resta da rede antes deste corte
            pct_remaining_before_cut = 1.0 - (step * absolute_step_chunk)
            
            # Rácio relativo a aplicar à camada encolhida para equivaler ao chunk absoluto
            current_step_ratio = absolute_step_chunk / pct_remaining_before_cut
            
            print(f"\n--- Roda Iterativa {step+1}/{num_steps} (A cortar +{absolute_step_chunk*100:.1f}% do total | Rácio relativo no passo: {current_step_ratio*100:.1f}%) ---")
            
            current_model = prune_fn(current_model, current_step_ratio, saliency_fn)
            
            # Capturar o Pré-FT inicial apenas no primeiríssimo corte para o histórico
            if step == 0 and 'pre_ft' in locals():
                pre_ft = evaluate_comprehensive(current_model, testloader, device)
                print(f"    Pré-FT Inicial → Acc: {pre_ft['accuracy']:.4f}")
            
            # Épocas adaptativas
            is_last_step = (step == num_steps - 1)
            step_epochs = epochs_per_step if is_last_step else max(1, epochs_per_step // 2)
            
            current_model = fine_tune(
                current_model, trainloader, valloader, 
                epochs=step_epochs, lr=lr, device=device
            )
            
    return current_model

### Loop de Experimentação

Compara sistematicamente:
- **Tipo de Poda:** Local L1 vs Global L1
- **Sparsity Ratios:** 10%, 20%, 30%, 50%, 70%

Para cada iteração executa: Poda → Fine-Tuning → Avaliação completa.

In [ ]:
# ── Configuração do Pipeline Experimental Atualizado ──

sparsity_ratios = [0.10, 0.20, 0.30, 0.50, 0.70]
fine_tune_epochs = 5
fine_tune_lr = 1e-4  

# Dicionário de métodos simplificado para passar apenas a função base
pruning_methods = {
    "Local L1 (Magnitude)": (prune_mlp_local, compute_l1_saliency),
    "Local L2 (Magnitude)": (prune_mlp_local, compute_l2_saliency),
    "Global L1 (Magnitude)": (prune_mlp_global, compute_l1_saliency),
    "Global L2 (Magnitude)": (prune_mlp_global, compute_l2_saliency),
}

estrategias = ["one-shot", "iterative"]

# ── Avaliação do Baseline ──
print("=" * 70)
print("AVALIAÇÃO DO MODELO BASELINE")
print("=" * 70)
baseline_results = evaluate_comprehensive(
    baseline_model, testloader, device
)
print(f"  Accuracy:   {baseline_results['accuracy']:.4f}")
print(f"  Parâmetros: {baseline_results['param_count']:,}") 
print(f"  Latência:   {baseline_results['latency_seconds']:.4f} s")
print(f"  VRAM:       {baseline_results['vram_mb']:.2f} MB")

# ── Armazenar todos os resultados ──
# Guardamos com uma chave descritiva para facilitar a criação da tabela e gráficos
all_results = {}

for method_name, (prune_fn, saliency_fn) in pruning_methods.items():
    for estrategia in estrategias:
        # Nome único combinado (ex: "Local L1 (Magnitude) (iterative)")
        full_method_key = f"{method_name} ({estrategia})"
        
        print(f"\n{'=' * 70}")
        print(f"MÉTODO: {full_method_key}")
        print(f"{'=' * 70}")
        
        method_results = []
        
        for ratio in sparsity_ratios:
            print(f"\n{'─' * 55}")
            print(f"  Sparsity Ratio Alvo: {ratio*100:.0f}%")
            print(f"{'─' * 55}")
            
            # Começar sempre com uma cópia fresca e limpa do baseline
            pruned = copy.deepcopy(baseline_model)
            
            if estrategia == "one-shot":
                print(f"  [One-Shot] A aplicar poda total de {ratio*100:.0f}%...")
                pruned = prune_fn(pruned, ratio, saliency_fn)
                
                # Avaliar imediatamente antes de treinar (Pré-FT)
                pre_ft = evaluate_comprehensive(pruned, testloader, device)
                print(f"    Pré-FT  → Acc: {pre_ft['accuracy']:.4f} | Params: {pre_ft['param_count']:,}")
                
                # Fine-tuning completo de recuperação
                pruned = fine_tune(
                    pruned, trainloader, valloader,
                    epochs=fine_tune_epochs, lr=fine_tune_lr,
                    optimizer_type="sgd", device=device
                )
                
            elif estrategia == "iterative":
                print(f"  [Iterative] A iniciar ciclos de poda progressiva...")
                num_steps = 3  # Dividido em 3 sub-passos (ex: para 30% vai a 10% -> 20% -> 30%) [cite: 62]
                ratios_per_step = [(ratio / num_steps) * i for i in range(1, num_steps + 1)]
                
                pre_ft = None
                
                for step, current_step_ratio in enumerate(ratios_per_step):
                    print(f"\n    -> Passo {step+1}/{num_steps} (Rácio Parcial: {current_step_ratio*100:.1f}%)")
                    pruned = prune_fn(pruned, current_step_ratio, saliency_fn)
                    
                    # Captura o Pré-FT apenas no primeiro corte para termos o ponto de partida do colapso
                    if step == 0:
                        pre_ft = evaluate_comprehensive(pruned, testloader, device)
                        print(f"    Pré-FT Inicial → Acc: {pre_ft['accuracy']:.4f} | Params: {pre_ft['param_count']:,}")
                    
                    # Épocas adaptativas: passos intermédios são mais curtos para estabilização, o último recupera a fundo [cite: 62, 63]
                    is_last_step = (step == num_steps - 1)
                    step_epochs = fine_tune_epochs if is_last_step else max(1, fine_tune_epochs // 2)
                    
                    pruned = fine_tune(
                        pruned, trainloader, valloader,
                        epochs=step_epochs, lr=fine_tune_lr,
                        optimizer_type="sgd", device=device
                    )
            
            # 4. Avaliar pós-fine-tuning completo na meta final
            post_ft = evaluate_comprehensive(pruned, testloader, device)
            print(f"\n  Pós-FT Final → Acc: {post_ft['accuracy']:.4f} | "
                  f"Params: {post_ft['param_count']:,} | "
                  f"Latência: {post_ft['latency_seconds']:.4f}s | "
                  f"VRAM: {post_ft['vram_mb']:.2f} MB")
            
            method_results.append({
                "sparsity_ratio": ratio,
                "pre_finetune": pre_ft,
                "post_finetune": post_ft,
            })
            
            # Libertar a VRAM da GPU para a próxima iteração não estoirar a memória [cite: 15]
            del pruned
            if device.type == "cuda":
                torch.cuda.empty_cache()
        
        # Guarda os resultados indexados pelo nome do método + estratégia
        all_results[full_method_key] = method_results

print(f"\n{'=' * 70}")
print("EXPERIMENTAÇÃO COMPLETA COM AMBAS AS ESTRATÉGIAS")
print(f"{'=' * 70}")

AVALIAÇÃO DO MODELO BASELINE
  Accuracy:   0.5732
  Parâmetros: 8,928,778
  Latência:   13.5562 s
  VRAM:       0.00 MB

MÉTODO: Local L1 (Magnitude) (one-shot)

───────────────────────────────────────────────────────
  Sparsity Ratio Alvo: 10%
───────────────────────────────────────────────────────
  [One-Shot] A aplicar poda total de 10%...
  Poda Local (ratio=0.10) nas camadas: [1, 5, 9]
    Camada[1]: 2048 → 1843 neurónios (removidos 205)
    Camada[5]: 1024 → 921 neurónios (removidos 103)
    Camada[9]: 512 → 460 neurónios (removidos 52)
    Pré-FT  → Acc: 0.5608 | Params: 7,797,041
  Fine-tuning: 5 epochs | LR=0.0001 | Optimizer=SGD")


    Epoch [1/5] Terminado -> Train Loss: 0.7894 | Val Loss: 0.5715 | Val Acc: 82.49%


    Epoch [2/5] Terminado -> Train Loss: 0.7857 | Val Loss: 0.5725 | Val Acc: 82.35%


    Epoch [3/5] Terminado -> Train Loss: 0.7879 | Val Loss: 0.5678 | Val Acc: 83.01%


    Epoch [4/5] Terminado -> Train Loss: 0.7780 | Val Loss: 0.5624 | Val Acc: 82.99%


...

    Epoch [2/2] Terminado -> Train Loss: 0.7907 | Val Loss: 0.5693 | Val Acc: 82.63%

    -> Passo 2/3 (Rácio Parcial: 20.0%)
  Poda Local (ratio=0.20) nas camadas: [1, 5, 9]
    Camada[1]: 1843 → 1474 neurónios (removidos 369)
    Camada[5]: 921 → 736 neurónios (removidos 185)
    Camada[9]: 460 → 368 neurónios (removidos 92)
  Fine-tuning: 2 epochs | LR=0.0001 | Optimizer=SGD")


    Epoch [1/2] Terminado -> Train Loss: 0.9474 | Val Loss: 0.7033 | Val Acc: 78.99%


    Epoch [2/2] Terminado -> Train Loss: 0.9295 | Val Loss: 0.6903 | Val Acc: 78.80%

    -> Passo 3/3 (Rácio Parcial: 30.0%)
  Poda Local (ratio=0.30) nas camadas: [1, 5, 9]
    Camada[1]: 1474 → 1031 neurónios (removidos 443)
    Camada[5]: 736 → 515 neurónios (removidos 221)
    Camada[9]: 368 → 257 neurónios (removidos 111)
  Fine-tuning: 5 epochs | LR=0.0001 | Optimizer=SGD")


    Epoch [1/5] Terminado -> Train Loss: 1.3105 | Val Loss: 1.0633 | Val Acc: 69.03%


    Epoch [2/5] Terminado -> Train Loss: 1.2861 | Val Loss: 1.0318 | Val Acc: 69.48%


### Tabela Resumo dos Resultados

In [ ]:
import pandas as pd
from IPython.display import display

# Preparar a lista de dicionários para o DataFrame
tabela_dados = []

# Extrair métricas do Baseline
baseline_params = baseline_results['param_count']
baseline_lat = baseline_results['latency_seconds']

# Adicionar a linha do Baseline
tabela_dados.append({
    "Método": "Baseline",
    "Sparsity (%)": 0,
    "Accuracy": round(baseline_results['accuracy'], 4),
    "Parâmetros": baseline_params,
    "Compressão (x)": 1.00,
    "Latência (s)": round(baseline_lat, 4),
    "VRAM (MB)": round(baseline_results['vram_mb'], 2)
})

# Iterar sobre os resultados guardados no dicionário all_results
for method_name, results in all_results.items():
    for r in results:
        post = r['post_finetune']
        compression = baseline_params / max(post['param_count'], 1)
        
        tabela_dados.append({
            "Método": method_name,
            "Sparsity (%)": int(r['sparsity_ratio'] * 100),
            "Accuracy": round(post['accuracy'], 4),
            "Parâmetros": post['param_count'],
            "Compressão (x)": round(compression, 2),
            "Latência (s)": round(post['latency_seconds'], 4),
            "VRAM (MB)": round(post['vram_mb'], 2)
        })

# Criar o DataFrame e ordenar para melhor visualização (opcional)
df_resultados = pd.DataFrame(tabela_dados)

# Aplicar um estilo para destacar o Baseline e alinhar o texto
estilo_tabela = df_resultados.style.set_properties(**{'text-align': 'center'}) \
    .set_table_styles([dict(selector='th', props=[('text-align', 'center')])]) \
    .highlight_max(subset=['Accuracy', 'Compressão (x)'], color='lightgreen') \
    .highlight_min(subset=['Latência (s)'], color='lightgreen')

display(estilo_tabela)

NameError: name 'baseline_results' is not defined

---
## 5. Avaliação Global e Visualização

Gráficos comparativos dos resultados experimentais:
1. **Sparsity Ratio vs. Test Accuracy** — Comparação da degradação de exatidão
2. **Sparsity Ratio vs. Inference Latency** — Redução de latência
3. **Compressão de Parâmetros vs. Speedup** — Eficiência da compressão

In [ ]:
# Gráficos


fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle(
    "Análise de Poda Estruturada: Local L1 vs Global L1",
    fontsize=16, fontweight='bold', y=1.02
)

colors = {"Local L1": "#2196F3", "Global L1": "#FF5722"}
markers = {"Local L1": "o", "Global L1": "s"}

baseline_acc = baseline_results['accuracy']
baseline_lat = baseline_results['latency_seconds']
baseline_par = baseline_results['param_count']

# ── Gráfico 1: Sparsity Ratio vs. Test Accuracy ──
ax1 = axes[0]
for method_name, results in all_results.items():
    ratios = [r['sparsity_ratio'] * 100 for r in results]
    accs = [r['post_finetune']['accuracy'] * 100 for r in results]
    ax1.plot(
        ratios, accs,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

ax1.axhline(
    y=baseline_acc * 100, color='#4CAF50',
    linestyle='--', linewidth=2, alpha=0.8,
    label='Baseline'
)
ax1.set_xlabel('Sparsity Ratio (%)', fontsize=12)
ax1.set_ylabel('Test Accuracy (%)', fontsize=12)
ax1.set_title('Degradação de Exatidão', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10, loc='lower left')
ax1.grid(True, alpha=0.3)
ax1.set_xticks([10, 20, 30, 50, 70])

# ── Gráfico 2: Sparsity Ratio vs. Inference Latency ──
ax2 = axes[1]
for method_name, results in all_results.items():
    ratios = [r['sparsity_ratio'] * 100 for r in results]
    lats = [r['post_finetune']['latency_seconds'] for r in results]
    ax2.plot(
        ratios, lats,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

ax2.axhline(
    y=baseline_lat, color='#4CAF50',
    linestyle='--', linewidth=2, alpha=0.8,
    label='Baseline'
)
ax2.set_xlabel('Sparsity Ratio (%)', fontsize=12)
ax2.set_ylabel('Latência de Inferência (s)', fontsize=12)
ax2.set_title('Redução de Latência', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks([10, 20, 30, 50, 70])

# ── Gráfico 3: Compressão de Parâmetros vs. Speedup ──
ax3 = axes[2]
for method_name, results in all_results.items():
    compressions = [
        baseline_par / max(r['post_finetune']['param_count'], 1)
        for r in results
    ]
    speedups = [
        baseline_lat / max(r['post_finetune']['latency_seconds'], 1e-9)
        for r in results
    ]
    ax3.plot(
        compressions, speedups,
        marker=markers[method_name],
        color=colors[method_name],
        linewidth=2.5, markersize=9,
        label=method_name
    )

    # Anotar cada ponto com a sparsity ratio
    for i, r in enumerate(results):
        ax3.annotate(
            f"{r['sparsity_ratio']*100:.0f}%",
            (compressions[i], speedups[i]),
            textcoords="offset points",
            xytext=(8, 5), fontsize=8, alpha=0.7
        )

ax3.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax3.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax3.set_xlabel('Compressão de Parâmetros (x)', fontsize=12)
ax3.set_ylabel('Speedup (x)', fontsize=12)
ax3.set_title('Compressão vs Speedup', fontsize=14, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pruning_analysis_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGráficos guardados em: pruning_analysis_mlp.png")

---
## Conclusão

Este notebook implementou e comparou duas estratégias de **Poda Estruturada** num MLP treinado no CIFAR-10:

| Aspeto | Local L1 | Global L1 |
|--------|----------|-----------|
| **Ranking** | Independente por camada | Unificado (rede inteira) |
| **Distribuição** | Uniforme entre camadas | Adaptativa (baseada em importância real) |
| **Flexibilidade** | Previsível | Mais eficiente em preservar capacidade |

### Próximos Passos
- Integrar saliências baseadas em gradientes (Taylor 1ª ordem, OBD) nos hooks definidos
- Aplicar poda iterativa (múltiplas rondas de poda + fine-tuning)
- Testar com arquiteturas convolucionais (VGG, ResNet)